# Qdrant: search a knowledge base while respecting its metadata

A support assistant should retrieve relevant articles, but only from the caller's organization and only from published content. Ranking by vector similarity alone is not enough: an excellent match from another tenant or an internal draft is still ineligible.

We will build a small local collection, attach article payloads, search with mandatory filters, evaluate authored relevance examples, update publication status, paginate records and reopen the persisted store. The notebook contains all explanations and code. **No Qdrant server, Docker, cloud account, embedding download or API key is needed after installation.**

The default representation is a local TF-IDF vector. It demonstrates vector and payload mechanics; it is not a pretrained semantic model. Qdrant local mode is useful for development and correctness checks, but these results do not benchmark server HNSW, distributed execution or production authorization.

## 1. Start with a local client and a clear storage boundary

Install `qdrant-client==1.15.1` with numpy, pandas, scikit-learn and joblib in the notebook kernel. Qdrant is a vector database; `qdrant-client` is its Python client and includes a local mode used here.

A collection defines a vector schema. A point has an ID, vector and optional payload. A payload is metadata such as tenant, category, publication status and source text. We keep persistent local data under `outputs/qdrant/local_store/`. Each full run creates a unique collection, preserving prior collections; summary exports are replaced.

Only one local client should hold the same storage directory at once. Close it before reopening or before another process accesses it.

In [ ]:
from pathlib import Path
import json, uuid, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from qdrant_client import QdrantClient, models
from IPython.display import display

OUT = Path('outputs/qdrant'); OUT.mkdir(parents=True, exist_ok=True)
STORE = OUT / 'local_store'
if 'client' in globals():
    client.close()
client = QdrantClient(path=str(STORE))
collection = 'help_articles_' + uuid.uuid4().hex[:10]
print({'python':platform.python_version(),'qdrant_client':version('qdrant-client'),'collection':collection})

## 2. Keep identity and eligibility separate from similarity

Our 20 fictional articles include published alpha content, one alpha draft, and four beta articles on overlapping topics. Numeric point IDs are unique; payloads include `title`, `text`, `category`, `tenant`, `status` and `revision`.

Predict which rows must be excluded for an alpha reader before inspecting any vector score. Article text can be nearly identical across tenants, so a score threshold cannot enforce tenant ownership. In a real service, derive the tenant from the authenticated caller rather than accepting an arbitrary tenant field supplied by an untrusted request.

In [ ]:
# Fictional help-center articles authored for this notebook; no scraped or private content.
base_articles = [
 ('Reset a password', 'Reset a forgotten password using the account recovery email link.', 'account'),
 ('Two factor recovery', 'Recover two factor authentication access with backup codes when your phone is lost.', 'account'),
 ('Change an email address', 'Change your account email address after confirming the new email inbox.', 'account'),
 ('Unlock an account', 'Unlock a locked account after too many failed login attempts.', 'account'),
 ('Download an invoice', 'Download a billing invoice and receipt from the payment history page.', 'billing'),
 ('Duplicate payment', 'Report a duplicate card charge when the same payment appears twice.', 'billing'),
 ('Refund status', 'Track a refund status and check when money returns to your card.', 'billing'),
 ('Update a payment card', 'Update an expired credit card before the next subscription payment.', 'billing'),
 ('Track a shipment', 'Track a shipment with the package tracking number and courier link.', 'delivery'),
 ('Change a delivery address', 'Change the delivery address before a package leaves the warehouse.', 'delivery'),
 ('Damaged package', 'Report a damaged package with photos of the parcel and broken items.', 'delivery'),
 ('Missing delivery', 'Investigate a missing delivery when tracking says delivered but the package is absent.', 'delivery'),
 ('Cancel a subscription', 'Cancel a recurring subscription to stop future renewal charges.', 'subscription'),
 ('Pause a subscription', 'Pause a subscription temporarily and resume the plan next month.', 'subscription'),
 ('Upgrade a plan', 'Upgrade your subscription plan to increase storage and team member limits.', 'subscription'),
 ('Internal refund draft', 'Refund money instantly without review; this fictional draft is not published policy.', 'billing')
]
records = []
for i,(title,text,category) in enumerate(base_articles):
    records.append({'id':1001+i, 'title':title, 'text':text, 'category':category,
                    'tenant':'alpha', 'status':'draft' if i == 15 else 'published', 'revision':1})
# A second fictional tenant reuses four topics. Metadata must distinguish ownership.
for i in [0,4,8,12]:
    row = dict(records[i]); row.update(id=2001+i, tenant='beta', title='Beta: '+row['title'])
    records.append(row)
docs = pd.DataFrame(records)
assert docs.id.is_unique and len(docs) == 20
queries = pd.DataFrame([
    {'query':'forgotten password recovery email', 'relevant_id':1001, 'split':'development'},
    {'query':'duplicate card charge payment twice', 'relevant_id':1006, 'split':'development'},
    {'query':'track shipment courier number', 'relevant_id':1009, 'split':'development'},
    {'query':'phone lost two factor backup codes', 'relevant_id':1002, 'split':'test'},
    {'query':'damaged parcel broken items photos', 'relevant_id':1011, 'split':'test'},
    {'query':'pause subscription resume next month', 'relevant_id':1014, 'split':'test'}
])

display(docs[['id','title','tenant','status','category']])
assert len(docs.query("tenant == 'alpha' and status == 'published'")) == 15

## 3. Fit one encoder and use it consistently for queries

TF-IDF learns a vocabulary and inverse-document-frequency weights from the article corpus. The same transform must encode incoming queries. We store dense float32 vectors for this small collection, even though TF-IDF is naturally sparse. At scale, consider sparse-vector support or an appropriate dense encoder rather than blindly densifying a huge vocabulary.

The vectorizer's default L2 normalization gives unit-length nonzero vectors. Qdrant's cosine metric supports cosine ranking. An unknown-only query has no lexical evidence, so our helper rejects it before search.

The alpha/beta metadata does not influence the encoder; it determines eligibility separately. A shared encoder trained on a multi-tenant corpus is a teaching simplification, not a recommendation for mixing sensitive tenant training data.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))
sparse = vectorizer.fit_transform(docs.title + ' ' + docs.text)
vectors = sparse.toarray().astype(np.float32)
DIMENSION = vectors.shape[1]
assert np.isfinite(vectors).all() and np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-5)

def encode_query(text):
    vector = vectorizer.transform([text]).toarray()[0].astype(np.float32)
    if np.linalg.norm(vector) == 0:
        raise ValueError('No known query terms; ask for a more specific query.')
    return vector.tolist()

try:
    encode_query('zzzzunknownword')
except ValueError as exc:
    print('Expected empty-evidence check:',exc)
print('Vector dimension:',DIMENSION)

## 4. Define a collection before inserting points

We create one unnamed dense vector per point using cosine distance. Vector dimensionality belongs to the collection schema; a new encoder with a different dimension requires a new compatible collection or schema design.

`upsert` inserts new IDs and replaces existing IDs. Treat an ID as the identity of a document revision policy, not a row position that can change during sorting. `wait=True` asks the operation to complete before we proceed with dependent checks.

Before running, predict the point count after inserting the same batch twice: it should remain 20, not become 40.

In [ ]:
client.create_collection(collection_name=collection,
    vectors_config=models.VectorParams(size=DIMENSION,distance=models.Distance.COSINE))
points = [models.PointStruct(id=int(row['id']),vector=vector.tolist(),
                             payload={key:value for key,value in row.items() if key != 'id'})
          for row,vector in zip(docs.to_dict(orient='records'),vectors)]
client.upsert(collection_name=collection,points=points,wait=True)
client.upsert(collection_name=collection,points=points,wait=True)
assert client.count(collection_name=collection,exact=True).count == 20
print(client.get_collection(collection_name=collection).config.params.vectors)

## 5. See why an unfiltered result can be ineligible

First query deliberately without filters and inspect payloads. The collection contains overlapping tenants and a draft whose wording can rank highly for a refund query. A plausible score cannot distinguish published policy from an unpublished claim.

This unfiltered call is a diagnostic within our fictional dataset. Application search should go through the filtered helper defined next. Local payload filtering is a correctness mechanism, not a complete authentication/authorization system.

In [ ]:
unfiltered = client.query_points(collection_name=collection,query=encode_query('refund money instantly'),
                                 limit=5,with_payload=True).points
display(pd.DataFrame([{'id':p.id,'score':p.score,'title':p.payload['title'],
                       'tenant':p.payload['tenant'],'status':p.payload['status']} for p in unfiltered]))

## 6. Apply required filters inside the retrieval request

The helper always requires the requested tenant and `status == published`, with an optional category filter. Filtering before ranking avoids the failure of fetching global top-k and discarding ineligible hits afterward.

`must` means all listed conditions must match. Other Boolean filter structures support alternatives and exclusions; use them according to the intended eligibility rule. The application should not allow callers to bypass mandatory filters.

Predict what remains for the same refund query under alpha/billing eligibility. Check every returned payload, not just the first hit.

In [ ]:
def eligible_filter(tenant,category=None):
    conditions = [models.FieldCondition(key='tenant',match=models.MatchValue(value=tenant)),
                  models.FieldCondition(key='status',match=models.MatchValue(value='published'))]
    if category is not None:
        conditions.append(models.FieldCondition(key='category',match=models.MatchValue(value=category)))
    return models.Filter(must=conditions)

def search(text,tenant='alpha',category=None,k=3):
    if k < 1:
        raise ValueError('k must be positive')
    hits = client.query_points(collection_name=collection,query=encode_query(text),
        query_filter=eligible_filter(tenant,category),limit=k,with_payload=True).points
    for point in hits:
        assert point.payload['tenant'] == tenant and point.payload['status'] == 'published'
        assert category is None or point.payload['category'] == category
    return hits

def show_hits(hits):
    return pd.DataFrame([{'id':p.id,'score':p.score,'title':p.payload['title'],
                          'category':p.payload['category'],'tenant':p.payload['tenant']} for p in hits])

display(show_hits(search('refund money instantly',category='billing')))
assert not search('refund money',tenant='nonexistent')
print('A tenant with no eligible articles correctly returns an empty result.')

## 7. Check the filtered ranking against a direct calculation

A nonempty response is not enough. Select the same eligible population in pandas, compute direct dot products on its normalized vectors, and compare returned top IDs and scores. This checks metric, encoder and filter consistency for a known query.

On this small lexical example, equal scores can create ties; compare scores and membership rather than imposing an undocumented tie order. The chosen query has a clear top result. Server approximate search may not exactly match all flat neighbors and needs its own recall assessment.

In [ ]:
text = 'duplicate card charge payment twice'
hits = search(text,category='billing',k=3)
mask = ((docs.tenant=='alpha') & (docs.status=='published') & (docs.category=='billing')).to_numpy()
positions = np.flatnonzero(mask)
manual_scores = vectors[positions] @ np.asarray(encode_query(text),dtype=np.float32)
expected = positions[np.argsort(-manual_scores)[:3]]
assert int(hits[0].id) == int(docs.iloc[expected[0]].id)
score_by_id = dict(zip(docs.iloc[positions].id.astype(int),manual_scores))
assert all(np.isclose(p.score,score_by_id[int(p.id)],atol=1e-5) for p in hits)
assert np.allclose(sorted([p.score for p in hits],reverse=True),sorted(manual_scores,reverse=True)[:3],atol=1e-5)
display(show_hits(hits))

## 8. Measure relevance and filter correctness separately

For each authored query, one article is designated relevant. Hit@3 measures whether it is retrieved; reciprocal rank rewards appearing earlier. The separate payload assertions check that every result is eligible. A system can score well on relevance while violating tenant isolation, or obey every filter while returning irrelevant text.

Development and test queries are separated before this evaluation. Six simple lexical examples are a smoke test, not a production quality estimate. Broaden coverage with paraphrases, multiple relevant articles, no-answer queries and changing document status.

In [ ]:
evaluation_rows = []
for row in queries.itertuples(index=False):
    hits = search(row.query)
    found = [int(p.id) for p in hits]
    rank = found.index(row.relevant_id)+1 if row.relevant_id in found else None
    evaluation_rows.append({'query':row.query,'split':row.split,'relevant_id':row.relevant_id,
                            'hit_at_3':int(rank is not None),'reciprocal_rank_at_3':0 if rank is None else 1/rank,
                            'eligible_results':all(p.payload['tenant']=='alpha' and p.payload['status']=='published' for p in hits)})
evaluation = pd.DataFrame(evaluation_rows)
assert evaluation.eligible_results.all()
display(evaluation)
display(evaluation.groupby('split')[['hit_at_3','reciprocal_rank_at_3']].mean())

## 9. Change publication status without changing the vector

A document can become ineligible even when its text is unchanged. Update the status payload for the alpha password article, verify that filtered search excludes it, then restore the published status. This demonstrates partial payload updates rather than re-embedding identical text.

The local check completes immediately with `wait=True`. In a production deployment, evaluate the actual consistency guarantees and replicas relevant to your read/write configuration. Do not use this local result as evidence about distributed propagation time.

In [ ]:
client.set_payload(collection_name=collection,payload={'status':'draft','revision':2},points=[1001],wait=True)
assert 1001 not in [p.id for p in search('forgotten password recovery email',k=20)]
changed = client.retrieve(collection_name=collection,ids=[1001],with_payload=True)[0]
assert changed.payload['status']=='draft' and changed.payload['revision']==2
client.set_payload(collection_name=collection,payload={'status':'published','revision':1},points=[1001],wait=True)
assert 1001 in [p.id for p in search('forgotten password recovery email')]
print('Eligibility changed with payload status, then was restored.')

## 10. Verify upsert and deletion on a temporary point

If a document's text changes, update both its vector and its payload using the same ID. Otherwise search can rank an old representation while displaying new text. For this demonstration, use an explicitly temporary ID so the authored evaluation corpus remains intact.

After deletion, verify both the count and direct retrieval. A successful operation acknowledgment should be followed by a check appropriate to the application.

In [ ]:
temporary_id = 999999
client.upsert(collection_name=collection,points=[models.PointStruct(id=temporary_id,
    vector=encode_query('refund status card'),payload={'title':'Temporary note','text':'refund status card',
    'tenant':'alpha','status':'draft','category':'billing','revision':1})],wait=True)
assert client.count(collection_name=collection,exact=True).count == 21
client.upsert(collection_name=collection,points=[models.PointStruct(id=temporary_id,
    vector=encode_query('payment invoice receipt'),payload={'title':'Revised temporary note','text':'payment invoice receipt',
    'tenant':'alpha','status':'draft','category':'billing','revision':2})],wait=True)
assert client.count(collection_name=collection,exact=True).count == 21
client.delete(collection_name=collection,points_selector=models.PointIdsList(points=[temporary_id]),wait=True)
assert not client.retrieve(collection_name=collection,ids=[temporary_id])
assert client.count(collection_name=collection,exact=True).count == 20
print('Temporary point inserted, replaced and removed.')

## 11. Paginate the whole eligible population for an audit

Similarity search ranks by a query; scrolling enumerates stored records. Use the returned offset rather than assuming a single response contains every point. Request payloads without vectors when auditing metadata to avoid unnecessary data transfer.

Predict the number of eligible alpha records and compare it with the final scroll count. Check uniqueness across pages so a pagination mistake cannot masquerade as complete coverage.

In [ ]:
audit = []; offset = None
while True:
    page,offset = client.scroll(collection_name=collection,scroll_filter=eligible_filter('alpha'),
                               limit=4,offset=offset,with_payload=True,with_vectors=False)
    audit.extend(page)
    if offset is None:
        break
assert len(audit) == 15 and len({p.id for p in audit}) == 15
assert all(p.payload['tenant']=='alpha' and p.payload['status']=='published' for p in audit)
display(pd.DataFrame([{'id':p.id,**p.payload} for p in audit])[['id','title','status']])

## 12. Reopen the store and check query compatibility

Persist the query encoder and a manifest containing collection name, dimension and representation description. Close the local client before opening the same path again. Compare every eligible ID and its score for a fixed query after reopening. Equal-score hits can change order, especially near a top-k boundary, so compare the full eligible set by ID instead of requiring an arbitrary tie order. Also verify the unique highest-scoring result.

In a production embedding workflow, also record the embedding-model revision, text preprocessing, chunking policy and distance metric. Two encoders can produce the same dimension while defining incompatible vector spaces; a dimension check alone does not prove compatibility.

In [ ]:
reference_hits = search('damaged parcel broken items photos', k=20)
reference_ids = [p.id for p in reference_hits]
reference_scores = [p.score for p in reference_hits]
joblib.dump(vectorizer,OUT / 'query_encoder.joblib')
manifest = {'collection':collection,'dimension':DIMENSION,'metric':'COSINE','representation':'TF-IDF word/bigram v1',
            'client_version':version('qdrant-client'),'point_count':20,'required_filters':['tenant','status=published']}
(OUT / 'manifest.json').write_text(json.dumps(manifest,indent=2))
client.close()
client = QdrantClient(path=str(STORE))
vectorizer = joblib.load(OUT / 'query_encoder.joblib')
assert len(vectorizer.vocabulary_) == DIMENSION
reopened_hits = search('damaged parcel broken items photos', k=20)
# Check the full eligible set by ID; equal-score hits need not retain their order.
reference_by_id = dict(zip(reference_ids, reference_scores))
reopened_by_id = {p.id: p.score for p in reopened_hits}
assert set(reopened_by_id) == set(reference_by_id)
assert all(np.isclose(reopened_by_id[point_id], score, atol=1e-6)
           for point_id, score in reference_by_id.items())
assert reopened_hits[0].id == reference_hits[0].id  # This query has a unique best match.
assert client.count(collection_name=collection,exact=True).count == manifest['point_count']
evaluation.to_csv(OUT / 'retrieval_evaluation.csv',index=False)
docs.to_json(OUT / 'source_articles.json',orient='records',indent=2)
client.close()
print('Persistent store reopened successfully; client closed.')

## 13. Know which conclusions require server mode

The local client supports a convenient subset of the database workflow. Server-side payload indexing and HNSW performance must be measured on an actual Qdrant deployment. Local-mode timings and payload-index behavior do not establish server search performance.

After configuring a server you control, the analogous connection is:

```python
server = QdrantClient(url='http://localhost:6333')
# Create/populate the intended collection on that server first.
server.create_payload_index(
    collection_name='your_collection',
    field_name='tenant',
    field_schema=models.PayloadSchemaType.KEYWORD,
)
server.close()
```

This illustrative snippet is not executed here. Remote credentials belong in environment/configuration facilities rather than notebook text. Payload indexes should match frequently used filter fields and their types. Benchmark representative filters, because a tenant-restricted search can behave differently from global search.

| Need | Concept to explore next | Question to verify |
|---|---|---|
| Dense semantic retrieval | Appropriate external embedding model | Does it handle domain language and paraphrases? |
| Keyword plus semantic search | Sparse vectors and hybrid queries | Does fusion improve held-out relevance? |
| Different representations per document | Named vectors | Does each query use the matching vector name? |
| Grouped results | Group by document ID | Are multiple chunks crowding out diverse documents? |
| Larger collections | HNSW, quantization and on-disk options | What recall, memory and latency are acceptable? |
| Operational recovery | Supported snapshots/backups | Can you restore and verify a real collection? |

Do not manually copy an active local database as though it were a verified server backup. This tutorial proves a clean close/reopen, not disaster recovery or high availability.

## 14. Explain what the evidence supports

1. Add a beta article whose text exactly matches an alpha query. Verify that the filtered alpha response still contains only alpha/published content.
2. Add a no-answer query composed of known corpus words. Explain why the engine can return a low-quality match despite nonzero lexical evidence. Calibrate any score threshold on labeled development data.
3. Design relevance judgments with multiple acceptable articles, then compute recall and precision with explicit denominators.
4. Replace TF-IDF with a reviewed sentence encoder in a new collection and compare on untouched queries. Record its model version and rebuild vectors consistently.

| Observation | Investigate |
|---|---|
| Wrong-tenant hit | Missing mandatory filter or untrusted tenant selection |
| No results | Eligible population, filter types and query vocabulary |
| Dimension error | Collection schema and encoder mismatch |
| Plausible but irrelevant hit | Lexical representation, missing context and judgment coverage |
| Local store reports a lock | An existing client/process still holds the directory |
| Updating text does not change ranking | Vector was not updated with the new text |

Official references: [Python client and local mode](https://github.com/qdrant/qdrant-client), [beginner retrieval](https://qdrant.tech/documentation/tutorials/search-beginners/), [payload filtering](https://qdrant.tech/documentation/search/filtering/), [point operations](https://qdrant.tech/documentation/concepts/points/), and [collections](https://qdrant.tech/documentation/concepts/collections/).

## Second project: Publish a complete knowledge-base revision

Use a local Qdrant instance to stage a fictional vector corpus. Inject malformed deliveries, batch ingestion, retry safely, and compare mixing revisions with querying a verified revision explicitly.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through identity and publication

A document ID identifies a logical article; a point ID here identifies an article within a revision. Keeping both revisions allows rollback. If revisions reuse point IDs, an upsert overwrites the old point and removes that rollback path. Use `revision * 10000 + document_id` for this bounded fixture, and validate the document range to avoid collisions.

A correct dimension does not prove a vector matches its article. Real ingestion also needs an encoder version and content hash. This fixture records an encoder tag and validates finite, nonzero vectors plus tenant identity before writing anything.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from qdrant_client import QdrantClient,models
p2_rng=np.random.default_rng(54)
p2_client=QdrantClient(':memory:')
p2_client.create_collection('revisions',vectors_config=models.VectorParams(size=16,distance=models.Distance.COSINE))
p2_v=p2_rng.normal(size=(240,16)).astype('float32');p2_v/=np.linalg.norm(p2_v,axis=1,keepdims=True)
p2_deliveries=[{'doc':i,'tenant':'alpha' if i%2==0 else 'beta','vector':v} for i,v in enumerate(p2_v)]
p2_deliveries += [p2_deliveries[0],{'doc':241,'tenant':None,'vector':p2_v[0]},{'doc':242,'tenant':'alpha','vector':np.zeros(16)}]
p2_clean={};p2_rejected=[];p2_duplicates=0
for r in p2_deliveries:
    v=np.asarray(r['vector'])
    if not 0<=r['doc']<10000 or r['tenant'] not in ['alpha','beta'] or v.shape!=(16,) or not np.isfinite(v).all() or np.linalg.norm(v)==0:
        p2_rejected.append(r['doc']);continue
    if r['doc'] in p2_clean:
        prior=p2_clean[r['doc']]
        if prior['tenant']!=r['tenant'] or not np.array_equal(prior['vector'],v):raise ValueError('Conflicting delivery')
        p2_duplicates+=1;continue
    p2_clean[r['doc']]=r
p2_calls=0
for revision in [1,2]:
    points=[models.PointStruct(id=revision*10000+r['doc'],vector=r['vector'].tolist(),payload={'doc':r['doc'],'tenant':r['tenant'],'revision':revision,'encoder':'synthetic-unit-v1'}) for r in p2_clean.values()]
    for start in range(0,len(points),64):
        p2_client.upsert('revisions',points=points[start:start+64],wait=True);p2_calls+=1
    # Replay a successful batch, as might happen after a lost acknowledgement.
    p2_client.upsert('revisions',points=points[:64],wait=True);p2_calls+=1
assert p2_client.count('revisions',exact=True).count==480
print({'rejected':p2_rejected,'duplicates':p2_duplicates,'batched_calls_including_retries':p2_calls,'one_point_calls_without_retries':480})

### Debug duplicate evidence before releasing the new revision

Both revisions deliberately have identical vectors so revision mistakes are isolated from semantic differences. Searching both can return the same logical document twice, reducing distinct evidence. Before switching the application to revision 2, count its tenant population and run filtered retrieval checks. The filter is part of the request contract, not a cleanup step after retrieval.

The local example does not implement a distributed atomic publication protocol. In production, store the active revision in a consistent release mechanism and retain revision 1 until readers and rollback requirements permit deletion.

In [ ]:
def p2_filter(revision=None):
    conditions=[models.FieldCondition(key='tenant',match=models.MatchValue(value='alpha'))]
    if revision is not None:conditions.append(models.FieldCondition(key='revision',match=models.MatchValue(value=revision)))
    return models.Filter(must=conditions)
p2_mixed=p2_client.query_points('revisions',query=p2_v[0].tolist(),query_filter=p2_filter(),limit=10).points
p2_rows=[]
for p2_revision in [1,2]:
    p2_count=p2_client.count('revisions',count_filter=p2_filter(p2_revision),exact=True).count
    assert p2_count==120
    p2_start=time.perf_counter()
    hits=p2_client.query_points('revisions',query=p2_v[0].tolist(),query_filter=p2_filter(p2_revision),limit=10).points
    assert len({p.payload['doc'] for p in hits})==10
    assert all(p.payload['tenant']=='alpha' and p.payload['revision']==p2_revision for p in hits)
    p2_rows.append({'revision':p2_revision,'eligible_count':p2_count,'distinct_documents':len({p.payload['doc'] for p in hits}),'query_seconds':time.perf_counter()-p2_start})
print({'mixed_revision_hits':len(p2_mixed),'mixed_distinct_documents':len({p.payload['doc'] for p in p2_mixed})})
display(pd.DataFrame(p2_rows));p2_client.close()

### Interpret cost and recovery

Batching reduces calls from 480 to eight plus two retries; it does not imply the same speedup over a remote network. Keeping two revisions doubles vector storage until retirement. Safe retries depend on stable IDs and deterministic payloads, which the population assertion checks here.

**Try:** omit the last batch of revision 2. **Worked reasoning:** the count gate should reject publication. A count alone can still pass with the wrong documents, so a stronger release check compares expected document IDs and content hashes as well as sample retrieval. Local mode is useful for correctness tests, not evidence of distributed-server throughput.